In [1]:
import sys
import os
from pathlib import Path
import pandas as pd
import torch
import numpy as np
import joblib
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments
)
# Если optimum падает с ошибкой, закомментируй пока ONNX часть, главное проверить обучение
from optimum.onnxruntime import ORTModelForSequenceClassification, ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig
from pymorphy3 import MorphAnalyzer
import nltk
from nltk.corpus import stopwords

# Настройка путей (ДЛЯ JUPYTER - указываем явно)
# Предполагаем, что ноутбук лежит в корне проекта или в папке scripts
BASE_DIR = Path(os.getcwd()).resolve().parent

# Проверка (должно вывести путь без 'notebooks' в конце)
print(f"Пути настроены. Базовая директория: {BASE_DIR}")

# Обновляем пути к данным (это оставь как есть, теперь заработает)
DATA_DIR = BASE_DIR / 'data'
TRAINING_SPECIFIC = DATA_DIR / 'TRAINING_DATASET_SPECIFIC.csv'
TRAINING_GENERAL = DATA_DIR / 'TRAINING_DATASET_GENERAL.csv'
HISTORY_LOG = DATA_DIR / 'history_log.csv'

# Пути к моделям
ONNX_SPECIFIC_DIR = BASE_DIR / 'onnx_models' / 'specific'
ONNX_GENERAL_DIR = BASE_DIR / 'onnx_models' / 'general'
ENCODER_SPECIFIC_PATH = BASE_DIR / 'label_encoder_specific.joblib'
ENCODER_GENERAL_PATH = BASE_DIR / 'label_encoder_general.joblib'

# Инит
try:
    stopwords.words("russian")
except LookupError:
    nltk.download('stopwords')
morph = MorphAnalyzer()
stopwords_ru = stopwords.words("russian")
MODEL_NAME = 'cointegrated/rubert-tiny'

print("Пути настроены. Базовая директория:", BASE_DIR)

W1211 18:54:25.264000 18876 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Пути настроены. Базовая директория: D:\projects\context\contextual_search
Пути настроены. Базовая директория: D:\projects\context\contextual_search


In [2]:
def preprocess_text(text: str) -> str:
    if pd.isna(text) or not text:
        return ""
    text = str(text).lower()
    text = re.sub(r'[^а-яa-z0-9\s]', ' ', text)
    words = text.split()
    lemmatized_words = [morph.parse(word)[0].normal_form for word in words]
    tokens = [token for token in lemmatized_words if token.strip() and token not in stopwords_ru]
    return " ".join(tokens)

print("Функция предобработки готова.")
# Тест
print(preprocess_text("Привет, это тестовое сообщение с PostgreSQL!"))

Функция предобработки готова.
привет это тестовый сообщение postgresql


In [3]:
if TRAINING_SPECIFIC.exists():
    df_spec = pd.read_csv(TRAINING_SPECIFIC)
    print(f"Specific dataset: {len(df_spec)} строк")
else:
    print("Ошибка! Нет основного датасета specific")

if HISTORY_LOG.exists():
    df_hist = pd.read_csv(HISTORY_LOG)
    print(f"History log: {len(df_hist)} строк")
    print(df_hist.head(3))
else:
    print("History log пока не создан (это нормально, если API еще не работало)")

Specific dataset: 26641 строк
History log: 15 строк
   2025-12-09T21:19:07.284993  1  \
0  2025-12-09T21:19:59.837716  2   
1  2025-12-09T21:20:07.377985  1   
2  2025-12-09T21:20:12.760535  3   

  Информационное сообщение: Задача операционного дня (ZOD).  \
0  Ресурсы: Нехватка оперативной памяти или актив...          
1  Внимание! Зафиксирована аномально высокая загр...          
2  Ошибка прикладного уровня. Передано в поддержк...          

                  Проблема связана с инфраструктурой  True  
0  Monitoring alert! На сервере app-prod-03 зафик...  True  
1  Авария 10005 ИС: Monitoring (Zabbix) Описание:...  True  
2  Пользователи массово жалуются на медленную раб...  True  


In [4]:
# label из history_log, если он там есть
df_specific_final = df_spec.copy() # Пока берем только основу

if HISTORY_LOG.exists() and not df_hist.empty:
    #только те, где pattern_found = True
    # В main.py мы писали: ["timestamp", "id", "standardized_text", "original_text", "pattern_found"]
    print("Проверка колонок:", df_hist.columns)
    # ТАк как в CSV нет чистого лейбла (pattern_name), дообучаться на нем сложно.
    # РЕШЕНИЕ НА СЕЙЧАС: Обучаться только на TRAINING_DATASET.csv
    # А history_log использовать только для аналитики.
    pass

print(f"Итого для обучения: {len(df_specific_final)}")

Проверка колонок: Index(['2025-12-09T21:19:07.284993', '1',
       'Информационное сообщение: Задача операционного дня (ZOD).',
       'Проблема связана с инфраструктурой', 'True'],
      dtype='object')
Итого для обучения: 26641


In [6]:
# Подготовка к обучению
X = df_specific_final['processed_text'].fillna('')
if 'processed_text' not in df_specific_final.columns:
    print("Выполняем препроцессинг...")
    df_specific_final['processed_text'] = df_specific_final['text'].apply(preprocess_text)

y = df_specific_final['label']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Сплит
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2)

# Токенизация
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_enc = tokenizer(X_train.tolist(), truncation=True, padding=True, max_length=128) # 128 для скорости теста
test_enc = tokenizer(X_test.tolist(), truncation=True, padding=True, max_length=128)

class TorchDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
        
    def __getitem__(self, idx):
        # ВАЖНО: приводим метки к torch.long
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
        
    def __len__(self):
        return len(self.labels)

train_ds = TorchDataset(train_enc, y_train)
test_ds = TorchDataset(test_enc, y_test)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(le.classes_))

# Аргументы обучения
args = TrainingArguments(
    output_dir="test_trainer",
    num_train_epochs=3,              # Ставим 3 эпохи для приличия (1 мало)
    per_device_train_batch_size=8,   # Если будет падать по памяти, уменьши до 4
    logging_steps=10,
    dataloader_num_workers=0,        # <--- ВАЖНО ДЛЯ WINDOWS (убирает ошибку мультипроцессинга)
    save_strategy="no",              # Не сохранять чекпоинты каждую секунду (экономим место)
    report_to="none"                 # Отключаем лишние репорты (wandb и т.д.)
)

trainer = Trainer(
    model=model, 
    args=args, 
    train_dataset=train_ds, 
    eval_dataset=test_ds
)

print("Запуск обучения...")
trainer.train()

# Оценка результата (чтобы показать цифры)
metrics = trainer.evaluate()
print(f"Результаты обучения: {metrics}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Запуск обучения...


D:\projects\context\contextual_search\venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,2.906400
20,2.846200
30,2.769500
40,2.722900
50,2.576700
60,2.468000
70,2.457300
80,2.318600
90,2.160600
100,2.107200


Результаты обучения: {'eval_loss': 0.0005062863347120583, 'eval_runtime': 8.8947, 'eval_samples_per_second': 599.119, 'eval_steps_per_second': 74.988, 'epoch': 3.0}
